# Clean rebuild PRONOSTIA - notebook-only

Mỗi ô xử lý một bước. Code và tests nằm trong data_process.ipynb ngay bên cạnh. Mở notebook này từ thư mục bàn giao, Restart Kernel, Run All. Không dùng cache cũ trong clean_rebuild; không tự xóa thư mục output. RAW_ROOT/OUTPUT_ROOT đặt tại ô cấu hình đường dẫn hoặc biến môi trường. Ground truth 11 bearings có nguồn trong test_ground_truth.csv; Full_Test_Set chỉ dùng kiểm chứng timestamp, không làm feature input.

**Google Colab:** mở notebook, nhận ZIP bàn giao ở các cell đầu, chọn raw download/Drive, rồi Run All. Xem COLAB_GUIDE.md trong ZIP.

### Import công cụ chuẩn bị Colab



In [ ]:
from pathlib import Path
import os
import sys
import json
import hashlib
import shutil
import subprocess
import importlib.util
import zipfile
import urllib.request
import uuid
from datetime import datetime, timezone


### Nhận diện môi trường Colab



In [ ]:
try:
    IS_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IS_COLAB = False
print('Runtime:', 'Google Colab' if IS_COLAB else 'Local Jupyter')


### Khai báo đường dẫn Colab

Mặc định dùng ổ /content. Để đọc Drive: bật COLAB_MOUNT_DRIVE; chọn drive_zip hoặc existing và sửa đường dẫn. Để giữ output trên Drive: đặt COLAB_OUTPUT_BASE=/content/drive/MyDrive/Bearing_RUL/outputs. Mỗi lần clean rebuild có tên output mới; không tự xóa dữ liệu.

In [ ]:
COLAB_MOUNT_DRIVE = False # @param {type:"boolean"}
COLAB_WORK_DIR = '/content/bearing_rul/handover_notebook_only' # @param {type:"string"}
COLAB_BUNDLE_ZIP = '/content/handover_notebook_only.zip' # @param {type:"string"}
COLAB_RAW_SOURCE = 'download' # @param ["download", "drive_zip", "existing"]
COLAB_RAW_ROOT = '/content/datasets/pronostia/ieee-phm-2012-data-challenge-dataset-master' # @param {type:"string"}
COLAB_OUTPUT_BASE = '/content/bearing_rul_outputs' # @param {type:"string"}
COLAB_DRIVE_ARCHIVE = '/content/drive/MyDrive/Bearing_RUL/phm2012-master.zip' # @param {type:"string"}
COLAB_DOWNLOAD_ARCHIVE = '/content/datasets/pronostia/phm2012-master.zip' # @param {type:"string"}


### Mount Google Drive nếu được chọn

Bạn tự xác nhận hộp cấp quyền Google Drive khi Colab yêu cầu. Chạy download mặc định không cần mount Drive.

In [ ]:
if IS_COLAB and (COLAB_MOUNT_DRIVE or COLAB_RAW_SOURCE=='drive_zip'):
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Không mount Drive; dùng filesystem hiện tại.')


### Chuẩn bị thư mục notebook trên Colab



In [ ]:
if IS_COLAB:
    Path(COLAB_WORK_DIR).mkdir(parents=True,exist_ok=True)
    os.chdir(COLAB_WORK_DIR)
print('Working directory:',Path.cwd())


### Nhận ZIP bộ notebook trên Colab

Nếu đã upload ZIP ở thanh Files của Colab, đặt COLAB_BUNDLE_ZIP đúng vị trí để bỏ qua hộp upload. Hai notebook và YAML/CSV phải có thật trên filesystem runtime; việc mở một notebook trong giao diện không tự tạo các file còn lại.

In [ ]:
required_bundle_files = ['data_process.ipynb','data_pipeline_scripts.ipynb','pronostia_config.yaml','test_ground_truth.csv','requirements.txt','README_NOTEBOOKS.md']
if IS_COLAB and not all(Path(name).is_file() for name in required_bundle_files):
    bundle_path = Path(COLAB_BUNDLE_ZIP)
    if not bundle_path.is_file():
        from google.colab import files
        print('Chọn handover_notebook_only.zip, KHÔNG chọn raw ZIP ở bước này.')
        uploaded = files.upload()
        candidates = [name for name in uploaded if name.lower().endswith('.zip')]
        assert len(candidates)==1, 'Cần đúng một ZIP bộ notebook'
        bundle_path = Path(candidates[0])
        bundle_path.write_bytes(uploaded[candidates[0]])
else:
    bundle_path = None


### Giải nén các file notebook cần thiết



In [ ]:
if IS_COLAB and bundle_path is not None:
    with zipfile.ZipFile(bundle_path) as bundle:
        for filename in required_bundle_files:
            candidates=[name for name in bundle.namelist() if name in (filename,'handover_notebook_only/'+filename)]
            assert len(candidates)==1, f'Thiếu/trùng file trong ZIP: {filename}'
            Path(filename).write_bytes(bundle.read(candidates[0]))
    print('Đã nhận đầy đủ notebook/config/dependencies. Bỏ qua reports cũ trong ZIP.')


### Cài dependency trên Colab

Cell chỉ cài packages trong requirements.txt, không nâng cấp hàng loạt thư viện Colab. Nếu runtime yêu cầu restart sau cài, restart rồi Run All; bộ file đã giải nén nên không cần upload lại.

In [ ]:
if IS_COLAB:
    subprocess.check_call([sys.executable,'-m','pip','install','--quiet','-r','requirements.txt'])
else:
    print('Local: dùng môi trường đã cài; pip install -r requirements.txt khi cần.')


### Import công cụ notebook

In [ ]:
from pathlib import Path
import os
import sys
import json
import platform
import uuid
from datetime import datetime
import nbformat


### Nạp định nghĩa từ data notebook

In [ ]:
data_path = Path.cwd() / 'data_process.ipynb'
assert data_path.is_file(), f'Thiếu notebook cùng thư mục: {data_path}'
data_notebook = nbformat.read(data_path,as_version=4)
definition_phases = {'imports','config','definition','test_definition'}
for cell in data_notebook.cells:
    if cell.cell_type=='code' and cell.metadata.get('phase') in definition_phases:
        exec(compile(cell.source,'data_process.ipynb: '+cell.metadata.get('task','definition'),'exec'),globals())


### Cấu hình đường dẫn và execution mode

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
RAW_ROOT = Path(os.environ.get('PRONOSTIA_RAW_ROOT') or CONFIG.get('raw_root') or
                (Path(COLAB_RAW_ROOT) if IS_COLAB else Path.home() / 'Downloads' / 'PHM_2012_Raw' / 'ieee-phm-2012-data-challenge-dataset-master')).resolve()
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:8]
OUTPUT_ROOT = Path(os.environ.get('PRONOSTIA_OUTPUT_ROOT') or CONFIG.get('output_root') or (Path(COLAB_OUTPUT_BASE)/RUN_ID if IS_COLAB else PROJECT_ROOT/'runs'/RUN_ID)).resolve()
EXECUTION_MODE = os.environ.get('PRONOSTIA_EXECUTION_MODE',CONFIG['execution_mode'])
REPORTS_ROOT = OUTPUT_ROOT/'reports' if IS_COLAB else PROJECT_ROOT/'reports'
RUN_CONFIG = copy.deepcopy(CONFIG)
RUN_CONFIG.update(raw_root=str(RAW_ROOT),output_root=str(OUTPUT_ROOT))
assert EXECUTION_MODE in ('clean_rebuild','validate_existing')
assert RUN_CONFIG['max_files'] is None


### Tải ZIP raw dataset đã pin

Source URL, commit và SHA256 đọc từ YAML. drive_zip đọc một ZIP từ Drive và giải nén về RAW_ROOT; nên để RAW_ROOT trên /content để giảm số lần I/O Drive. existing dùng raw đã giải nén.

In [ ]:
if IS_COLAB and COLAB_RAW_SOURCE=='download' and not RAW_ROOT.exists():
    raw_archive = Path(COLAB_DOWNLOAD_ARCHIVE)
    if not raw_archive.exists():
        raw_archive.parent.mkdir(parents=True,exist_ok=True)
        partial=raw_archive.with_suffix('.zip.part')
        with urllib.request.urlopen(CONFIG['dataset_source']['archive_url'],timeout=120) as response, partial.open('wb') as target:
            shutil.copyfileobj(response,target,length=4*1024*1024)
        partial.replace(raw_archive)
    print('Archive:',raw_archive)
elif IS_COLAB and COLAB_RAW_SOURCE=='drive_zip' and not RAW_ROOT.exists():
    raw_archive=Path(COLAB_DRIVE_ARCHIVE)
    assert raw_archive.is_file(), f'Upload raw ZIP vào Drive trước: {raw_archive}'
else:
    raw_archive=None


### Xác minh checksum và commit raw ZIP



In [ ]:
archive_verification=None
if raw_archive is not None:
    archive_verification={'sha256':verify_pinned_archive(raw_archive,CONFIG['dataset_source']),
                          'commit_sha':CONFIG['dataset_source']['commit_sha'],'status':'PASS'}
    print(archive_verification)


### Giải nén raw vào thư mục mới



In [ ]:
if raw_archive is not None:
    assert archive_verification['status']=='PASS'
    assert not RAW_ROOT.exists(), 'Không ghi đè raw đang có'
    stage=RAW_ROOT.parent/(RAW_ROOT.name+'.extracting_'+uuid.uuid4().hex[:8])
    stage.mkdir(parents=True)
    with zipfile.ZipFile(raw_archive) as archive:
        top_levels={name.split('/')[0] for name in archive.namelist() if name}
        assert len(top_levels)==1, 'Archive phải có đúng một thư mục gốc'
        for member in archive.infolist():
            parts=Path(member.filename).parts[1:]
            if not parts:continue
            destination=(stage.joinpath(*parts)).resolve()
            assert destination.is_relative_to(stage.resolve()), 'Unsafe archive path'
            if member.is_dir():destination.mkdir(parents=True,exist_ok=True)
            else:
                destination.parent.mkdir(parents=True,exist_ok=True)
                with archive.open(member) as source,destination.open('wb') as target:shutil.copyfileobj(source,target)
    stage.rename(RAW_ROOT)
    print('Raw extracted:',RAW_ROOT)


### Kiểm tra đường dẫn raw cho toàn bộ pipeline



In [ ]:
assert RAW_ROOT.is_dir(), f'Đặt RAW_ROOT/PRONOSTIA_RAW_ROOT: {RAW_ROOT}'
assert all((RAW_ROOT/subset).is_dir() for subset in ('Learning_set','Test_set','Full_Test_Set')), 'Raw cần đủ ba tập'
print('Raw dataset path: PASS')


### Hiển thị môi trường chạy hiện tại

In [ ]:
ENVIRONMENT = {'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,
               'torch':torch.__version__,'scipy':__import__('scipy').__version__,'yaml':yaml.__version__,'platform':platform.platform(),'runtime':'google_colab' if IS_COLAB else 'local',
               'started_at_utc':datetime.now(timezone.utc).isoformat(),
               'colab_release_tag':os.environ.get('COLAB_RELEASE_TAG'),
               'colab_module_available':IS_COLAB}
print('PROJECT_ROOT:',PROJECT_ROOT)
print('RAW_ROOT:',RAW_ROOT)
print('OUTPUT_ROOT:',OUTPUT_ROOT)
display(pd.DataFrame([ENVIRONMENT]))


### Đặt random seed

In [ ]:
np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])


### Giới hạn CPU threads

In [ ]:
torch.set_num_threads(2)


### Kiểm tra output directory

In [ ]:
if EXECUTION_MODE=='clean_rebuild':
    if OUTPUT_ROOT.exists() and any(OUTPUT_ROOT.iterdir()):
        raise FileExistsError('clean_rebuild yêu cầu output mới/rỗng; không tự xóa cache cũ')
    OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
else:
    assert (OUTPUT_ROOT/'manifest_all_files.csv').is_file(), 'validate_existing cần output cache hợp lệ'
REPORTS_ROOT.mkdir(exist_ok=True)


### Chuẩn bị fixture tests

In [ ]:
Path('.tmp').mkdir(exist_ok=True)


### Chạy parser tests

In [ ]:
parser_test_result = execute_test_classes([ParserTests])


### Chạy contract tests

In [ ]:
contract_test_result = execute_test_classes([ContractTests])


### Chạy identity tests

In [ ]:
identity_test_result = execute_test_classes([CacheIdentityTests])


### Chạy feedback tests

In [ ]:
feedback_test_result = execute_test_classes([FeedbackRegressionTests])


### Chạy integration tests

In [ ]:
integration_test_result = execute_test_classes([FixtureIntegrationTests])


### Tổng hợp test results

In [ ]:
TEST_RESULTS = {'parser':parser_test_result, 'contract':contract_test_result, 'identity':identity_test_result, 'feedback':feedback_test_result, 'integration':integration_test_result}
display(pd.DataFrame.from_dict(TEST_RESULTS,orient='index'))


### Nạp ground truth có nguồn

In [ ]:
GROUND_TRUTH = pd.read_csv(PROJECT_ROOT/'test_ground_truth.csv')
assert set(GROUND_TRUTH.bearing_id)==set(TEST)
assert GROUND_TRUTH.source.notna().all() and GROUND_TRUTH.source_locator.notna().all()
assert dict(zip(GROUND_TRUTH.bearing_id,GROUND_TRUTH.rul_seconds))==RUN_CONFIG['test_rul_at_cut_seconds']


### Scan cấu trúc dataset

In [ ]:
tasks, structure_issues = scan_dataset_structure(RAW_ROOT)
assert not any(i['severity']=='ERROR' for i in structure_issues),structure_issues


### Scan schema và checksum files

In [ ]:
records = scan_inventory_files(RAW_ROOT,tasks,RUN_CONFIG)
print('Raw CSV:',len(records))


### Validate filename indices

In [ ]:
index_issues = validate_filename_indices(records)


### Validate learning terminal completeness

In [ ]:
validate_learning_terminals(records,RUN_CONFIG['expected_learning_terminal_indices'])


### Build acquisition manifest

In [ ]:
manifest, clock_repairs = build_acquisition_records(records,RUN_CONFIG)


### Analyze data quality

In [ ]:
quality = analyze_inventory_quality(records,structure_issues,index_issues,clock_repairs)
assert not any(i['severity']=='ERROR' for i in quality['issues']),quality['issues']


### Export inventory manifests

In [ ]:
if EXECUTION_MODE=='clean_rebuild':
    pd.DataFrame(records).to_csv(OUTPUT_ROOT/'manifest_all_files.csv',index=False)
    manifest.to_csv(OUTPUT_ROOT/'manifest.csv',index=False)


### Export source recipe

In [ ]:
if EXECUTION_MODE=='clean_rebuild':
    (OUTPUT_ROOT/'preprocessing.yaml').write_text(yaml.safe_dump(RUN_CONFIG,sort_keys=True),encoding='utf-8')


### Compare ground truth with Full_Test_Set

In [ ]:
rul_audit = ground_truth_comparison(manifest,GROUND_TRUTH)
display(rul_audit)


### Export ground truth audit

In [ ]:
rul_audit.to_csv(REPORTS_ROOT/'ground_truth_full_test_comparison.csv',index=False)


### Select model input bearings

In [ ]:
frames = bearing_frames(manifest)
assert not any(f.subset.iloc[0]=='Full_Test_Set' for f in frames.values())


### Create absolute RUL labels

In [ ]:
label_frames = {b:label_bearing(frame,RUN_CONFIG) for b,frame in frames.items()}


### Load temperature streams

In [ ]:
streams = {b:load_bearing_temperature(RAW_ROOT,records,frame,RUN_CONFIG) for b,frame in label_frames.items()}


### Align causal temperature

In [ ]:
aligned = {b:align_bearing_temperature(streams[b],frame,RUN_CONFIG) for b,frame in label_frames.items()}


### Validate temperature alignment

In [ ]:
for values in aligned.values(): validate_temperature_alignment(values,RUN_CONFIG)


### Bổ sung trace nhãn vào manifest

In [ ]:
for b,frame in label_frames.items():
    rows=manifest.index[(manifest.bearing_id==b)&(manifest.subset==frame.subset.iloc[0])]
    assert len(rows)==len(frame)
    manifest.loc[rows,'rul_seconds']=frame.rul_seconds.to_numpy()
    manifest.loc[rows,'rul_cycles']=frame.rul_cycles.to_numpy()
    manifest.loc[rows,'temp_timestamp']=aligned[b]['matched']
    manifest.loc[rows,'alignment_delta_s']=aligned[b]['delta']
    manifest.loc[rows,'temp_missing_mask']=aligned[b]['temperature_mask'].reshape(len(frame),-1).all(axis=1)


### Export enriched acquisition manifest

In [ ]:
if EXECUTION_MODE=='clean_rebuild':
    manifest.to_csv(OUTPUT_ROOT/'manifest.csv',index=False)


### Transform vibration bearings

In [ ]:
vibration = {b:transform_bearing_vibration(RAW_ROOT,frame,RUN_CONFIG) for b,frame in label_frames.items()}


### Assemble acquisition shards

In [ ]:
arrays = {b:assemble_bearing_shard(frame,vibration[b],aligned[b]) for b,frame in label_frames.items()}


### Export acquisition shards

In [ ]:
shards = export_acquisition_shards(OUTPUT_ROOT,arrays) if EXECUTION_MODE=='clean_rebuild' else {b:OUTPUT_ROOT/'acquisitions'/f'{b}.npz' for b in arrays}


### Calculate temperature coverage

In [ ]:
coverage = temperature_coverage(records,label_frames,streams,aligned)
display(coverage)


### Export temperature coverage

In [ ]:
coverage.to_csv(REPORTS_ROOT/'temperature_coverage.csv',index=False)


### Analyze missing modality by fold

In [ ]:
split_protocols = protocols(RUN_CONFIG['seed'])
splits = split_protocols['development_cv']+[split_protocols['official_benchmark']]
missing_bearings = coverage.loc[coverage.high_warning,'bearing_id'].tolist()
fold_missing = [{'fold':str(s['fold']),'train_missing_temperature':[b for b in s['train'] if b in missing_bearings],
                'validation_missing_temperature':[b for b in s['validation'] if b in missing_bearings],
                'test_missing_temperature':[b for b in s['test'] if b in missing_bearings]} for s in splits]
display(pd.DataFrame(fold_missing))


### Export missing modality protocol

In [ ]:
write_json(REPORTS_ROOT/'modality_protocol.json',{'folds':fold_missing,'fully_missing_bearings':missing_bearings,
 'official_test_fully_missing':int(((coverage.subset=='Test_set')&coverage.high_warning).sum()),
 'official_test_partially_missing':int(((coverage.subset=='Test_set')&(coverage.missing_element_percent>0)&~coverage.high_warning).sum()),
 'policy':'Retain all bearings with per-element mask; do not fabricate temperature. Compare vibration-only and multimodal on identical bearing folds/windows/endpoint ground truth. Additional both-modality subset must list included bearings and reuse train-only normalization.'})


### Create temperature report

In [ ]:
temperature_report = make_temperature_report(coverage,aligned)


### Create label report

In [ ]:
label_report = make_label_report(label_frames)


### Export label and temperature reports

In [ ]:
write_json(REPORTS_ROOT/'label_report.json',label_report)
write_json(REPORTS_ROOT/'temperature_report.json',temperature_report)


### Export split protocols

In [ ]:
if EXECUTION_MODE=='clean_rebuild':
    (OUTPUT_ROOT/'splits.yaml').write_text(yaml.safe_dump(split_protocols,sort_keys=True),encoding='utf-8')


### Resolve fold folders

In [ ]:
folders = {str(s['fold']):OUTPUT_ROOT/('official' if s['fold']=='official' else f"fold_{s['fold']}") for s in splits}


### Nạp hoặc fit train-only fold scalers

In [ ]:
scalers = {str(s['fold']):fit_scaler(shards,s['train'],fingerprint(RUN_CONFIG),fingerprint(s),RUN_CONFIG['scaler_epsilon']) if EXECUTION_MODE=='clean_rebuild' else json.loads((folders[str(s['fold'])]/'scaler.json').read_text()) for s in splits}


### Export fold scaler provenance

In [ ]:
if EXECUTION_MODE=='clean_rebuild':
    for split in splits:
        folder=folders[str(split['fold'])];folder.mkdir(exist_ok=True)
        write_json(folder/'scaler.json',scalers[str(split['fold'])])
        write_json(folder/'split.json',split)


### Export normalized fold shards

In [ ]:
if EXECUTION_MODE=='clean_rebuild':
    for split in splits:
        export_scaled_shards(folders[str(split['fold'])],scale_fold_shards(shards,split,scalers[str(split['fold'])]))


### Export window metadata

In [ ]:
if EXECUTION_MODE=='clean_rebuild':
    for split in splits:export_window_metadata(folders[str(split['fold'])],split,RUN_CONFIG)


### Nạp hoặc build fold cache indices

In [ ]:
cache_indices = {str(s['fold']):build_cache_index(folders[str(s['fold'])],s) if EXECUTION_MODE=='clean_rebuild' else pd.read_csv(folders[str(s['fold'])]/'cache_index.csv') for s in splits}


### Export fold cache indices

In [ ]:
if EXECUTION_MODE=='clean_rebuild':
    for key,index in cache_indices.items():
        index.to_csv(folders[key]/'cache_index.csv',index=False)
        index.to_parquet(folders[key]/'cache_index.parquet',index=False)


### Export fold manifests

In [ ]:
if EXECUTION_MODE=='clean_rebuild':
    for split in splits:fold_manifest(manifest,split).to_csv(folders[str(split['fold'])]/'manifest.csv',index=False)


### Export cache identities

In [ ]:
if EXECUTION_MODE=='clean_rebuild':
    for split in splits:write_json(folders[str(split['fold'])]/'cache_identity.json',cache_identity(quality['source_hash'],RUN_CONFIG,split,scalers[str(split['fold'])]))


### Validate cache identities

In [ ]:
for folder in folders.values():verify_identity(folder,expected_config=RUN_CONFIG,verified_source_hash=quality['source_hash'])


### Validate split and scaler invariants

In [ ]:
for folder in folders.values():validate_split_scaler(folder)


### Validate shard labels and clocks

In [ ]:
validated_shards={}
for split in splits:
    key=str(split['fold'])
    for role in ('train','validation','test'):
        for b in split[role]:
            validate_one_shard(folders[key]/'shards'/role/f'{b}.npz',scalers[key]['hash'],role!='test')


### Validate window boundaries

In [ ]:
for split in splits:
    for role in ('train','validation','test'):
        for b in split[role]:
            with np.load(folders[str(split['fold'])]/'shards'/role/f'{b}.npz',allow_pickle=False) as z:
                validate_window_shard(z,role,RUN_CONFIG)


### Validate scaled temperature masks

In [ ]:
for split in splits:
    for role in ('train','validation','test'):
        for b in split[role]:
            with np.load(folders[str(split['fold'])]/'shards'/role/f'{b}.npz',allow_pickle=False) as z:validate_temperature_shard(z)


### Validate full-pass DataLoaders

In [ ]:
loader_results={str(s['fold']):{role:validate_one_loader(folders[str(s['fold'])],role,RUN_CONFIG)
                 for role in ('train','validation','test') if s[role]} for s in splits}


### Benchmark DataLoader batches

In [ ]:
benchmarks={str(s['fold']):{role:benchmark_loader(folders[str(s['fold'])],role)
             for role in ('train','validation','test') if s[role]} for s in splits}


### Aggregate subset-condition-split counts

In [ ]:
fold_counts = {str(s['fold']):aggregate_fold_counts(records,manifest,cache_indices[str(s['fold'])],s) for s in splits}


### Measure process RAM

In [ ]:
ram = {'rss_mb':psutil.Process().memory_info().rss/2**20}


### Assemble validation report

In [ ]:
validation_report={'status':'PASS','errors':[], 'folds':{key:{'status':'PASS','full_pass':loader_results[key],
 'benchmarks':benchmarks[key],'counts':fold_counts[key]} for key in folders},'environment':ENVIRONMENT,'ram':ram,
 'high_warnings':{'temperature_missing':missing_bearings,'ground_truth_full_test_mismatches':rul_audit.loc[~rul_audit.verified_against_full_test,'bearing_id'].tolist()},
 'official_benchmark_ground_truth_ready':bool(rul_audit.verified_against_full_test.all())}


### Export validation evidence

In [ ]:
write_json(REPORTS_ROOT/'validation_report.json',validation_report)
for key,value in validation_report['folds'].items():write_json(folders[key]/'validation_report.json',value)


### Export quality report

In [ ]:
write_json(REPORTS_ROOT/'data_quality_report.json',quality)


### Select learning-only QC samples

In [ ]:
qc_samples = select_qc_samples(RAW_ROOT,manifest,RUN_CONFIG)


### Visualize waveform

In [ ]:
waveform_figure = plot_waveform_qc(qc_samples)
display(waveform_figure)


### Export waveform figure

In [ ]:
waveform_figure.savefig(REPORTS_ROOT/'waveform.png',dpi=150)
plt.close(waveform_figure)


### Visualize psd

In [ ]:
psd_figure = plot_psd_qc(qc_samples)
display(psd_figure)


### Export psd figure

In [ ]:
psd_figure.savefig(REPORTS_ROOT/'psd.png',dpi=150)
plt.close(psd_figure)


### Visualize rms

In [ ]:
rms_figure = plot_rms_qc(shards)
display(rms_figure)


### Export rms figure

In [ ]:
rms_figure.savefig(REPORTS_ROOT/'rms.png',dpi=150)
plt.close(rms_figure)


### Visualize temperature_histogram

In [ ]:
temperature_histogram_figure = plot_temperature_histogram(streams)
display(temperature_histogram_figure)


### Export temperature_histogram figure

In [ ]:
temperature_histogram_figure.savefig(REPORTS_ROOT/'temperature_histogram.png',dpi=150)
plt.close(temperature_histogram_figure)


### Visualize alignment_histogram

In [ ]:
alignment_histogram_figure = plot_alignment_histogram(aligned)
display(alignment_histogram_figure)


### Export alignment_histogram figure

In [ ]:
alignment_histogram_figure.savefig(REPORTS_ROOT/'alignment_histogram.png',dpi=150)
plt.close(alignment_histogram_figure)


### Calculate anti-alias QC

In [ ]:
aliasing_qc = calculate_aliasing_qc(RUN_CONFIG['vibration'])


### Validate anti-alias invariant

In [ ]:
assert aliasing_qc['pass'],aliasing_qc


### Export preprocessing QC

In [ ]:
write_json(REPORTS_ROOT/'preprocessing_qc.json',{'aliasing_check':aliasing_qc,'figures':['waveform.png','psd.png','rms.png','temperature_histogram.png','alignment_histogram.png']})


### Display fold summary

In [ ]:
fold_summary=[{'fold':key,'status':'PASS',**{role:loader_results[key].get(role,{}).get('windows',0) for role in ('train','validation','test')}} for key in folders]
display(pd.DataFrame(fold_summary))


### Display official counts

In [ ]:
display(pd.DataFrame(fold_counts['official']))


### Assemble clean rebuild summary

In [ ]:
execution_summary={'preprocessing_status':'PASS','rebuilt':EXECUTION_MODE=='clean_rebuild','execution_mode':EXECUTION_MODE,
 'raw_root':str(RAW_ROOT),'output_root':str(OUTPUT_ROOT),'environment':ENVIRONMENT,'tests':TEST_RESULTS,'folds':fold_summary,
 'source_hash':quality['source_hash'],'config_hash':fingerprint(RUN_CONFIG),'dataset_source':RUN_CONFIG['dataset_source'],
 'ground_truth_comparison':rul_audit.to_dict('records'),'official_benchmark_ground_truth_ready':bool(rul_audit.verified_against_full_test.all()),
 'acceptance_status':'REVIEW_REQUIRED_GROUND_TRUTH' if not rul_audit.verified_against_full_test.all() else 'PASS',
 'modality_missing_bearings':missing_bearings,'aliasing_check':aliasing_qc}


### Export clean rebuild proof

In [ ]:
write_json(REPORTS_ROOT/'clean_rebuild_execution.json',execution_summary)
write_json(REPORTS_ROOT/'notebook_test_report.json',TEST_RESULTS)


### Export compact manifest summary

In [ ]:
pd.DataFrame(records).groupby(['subset','bearing_id','condition_id','modality'],sort=True).size().rename('file_count').reset_index().to_csv(REPORTS_ROOT/'manifest_summary.csv',index=False)


### Show final acceptance status

In [ ]:
print('preprocessing_status:',execution_summary['preprocessing_status'])
print('rebuilt:',execution_summary['rebuilt'])
print('tests:',sum(row['tests'] for row in TEST_RESULTS.values()))
print('acceptance_status:',execution_summary['acceptance_status'])
print('official_benchmark_ground_truth_ready:',execution_summary['official_benchmark_ground_truth_ready'])
print('Reports:',REPORTS_ROOT)


### Tạo báo cáo execution Google Colab



In [ ]:
if IS_COLAB:
    colab_report={**execution_summary,'report_kind':'actual_google_colab_execution','runtime':'google_colab',
        'finished_at_utc':datetime.now(timezone.utc).isoformat(),'archive_verification':archive_verification,
        'notebook':'data_pipeline_scripts.ipynb','test_count':sum(r['tests'] for r in TEST_RESULTS.values()),
        'validation_status':validation_report['status'],'full_pass':loader_results,
        'notebook_source_sha256':{name:hashlib.sha256((PROJECT_ROOT/name).read_bytes()).hexdigest()
                                  for name in ('data_process.ipynb','data_pipeline_scripts.ipynb')}}
    write_json(REPORTS_ROOT/'colab_execution.json',colab_report)
    print('Actual Colab execution report:',REPORTS_ROOT/'colab_execution.json')
else:
    print('Local execution; không ghi colab_execution.json.')


### Đóng gói reports của lượt chạy Colab

ZIP này chỉ chứa reports mới của runtime, không chứa cache lớn. Output trên /content mất khi runtime bị xóa; tải ZIP và notebook có output, hoặc dùng OUTPUT_BASE trên Google Drive để lưu bền.

In [ ]:
if IS_COLAB:
    data_proof=PROJECT_ROOT/'reports/data_notebook_colab_execution.json'
    if data_proof.exists():shutil.copy2(data_proof,REPORTS_ROOT/data_proof.name)
    report_zip=Path(shutil.make_archive(str(OUTPUT_ROOT/'colab_reports'),'zip',root_dir=REPORTS_ROOT))
    print('Reports ZIP:',report_zip)


### Tải báo cáo execution Colab về máy



In [ ]:
if IS_COLAB:
    from google.colab import files
    files.download(str(report_zip))
